# Lesson 01 - AI 代理入門

歡迎來到 **AI 代理初學者** 課程的第一課！

**AI 代理** 是一種使用大型語言模型 (LLM) 作為其推理引擎的程式，並且能在現實世界中採取*行動*——呼叫 API、查詢資料庫或執行程式碼——以代表使用者達成目標。

在這個筆記本中，您將建立第一個代理：一個推薦度假目的地的 **旅遊代理**。在此過程中您將學習如何：

1. 使用 **Microsoft Agent Framework** 連接到 Azure AI Foundry Agent 服務。
2. 給代理一個 **工具**——一個它可以呼叫的普通 Python 函式。
3. 執行代理並檢查其回應。
4. 逐字元串流代理的回應。


## 設定

在執行此筆記本之前，請確保您已經：

1. **擁有一個已部署聊天模型的 Azure AI Foundry 專案**（例如 `gpt-4o-mini`）。
2. **已使用 Azure CLI 登入** — 在終端機執行 `az login`。
3. **設定所需的環境變數：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Azure AI Foundry 專案端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您部署的模型名稱。

以下儲存格會安裝您需要的 Python 套件。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 建立您的第一個代理

代理需要兩樣東西：

- **指令**，告訴它*它是誰*以及*如何行為*（系統提示）。
- **工具** — 用 `@tool` 裝飾的 Python 函數，代理可以調用這些工具來檢索信息或執行操作。

下面我們定義了一個簡單的工具，返回熱門旅遊目的地列表。當用戶要求旅遊推薦時，代理將使用此工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 流式回應

為了更具互動性的體驗，你可以**流式**傳送代理的回應。代理不需要等候完整的回覆，而是隨著文字生成即時產出。這在聊天介面中特別有用，可以即時顯示輸出內容。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

在本課程中您學會了如何：

- **建立一個提供者**，透過 `AzureAIProjectAgentProvider` 連接到 Azure AI Foundry Agent Service。
- 使用 `@tool` 裝飾器 **定義工具**，以便代理能呼叫您的 Python 函數。
- 使用用戶訊息 **執行代理** 並列印其回應。
- **串流回應** 以取得即時輸出。

在下一課中，我們將更深入探討代理框架，並學習如何為代理提供更強大的工具和多步推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件乃使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們致力於確保翻譯準確性，但請注意，自動翻譯可能包含錯誤或不準確之處。文件原文以其原始語言版本為權威依據。對於重要資訊，建議採用專業人工翻譯。本公司概不對因使用本翻譯所引致之任何誤解或誤釋負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
